In [17]:
import emcee
import h5py
import numpy as np
from pathlib import Path
from scipy.special import lambertw
from numpy.random import default_rng
from scipy import linalg
from scipy.signal import argrelextrema
from scipy.linalg import sqrtm
import PPmodel as pp
import torch
import sys
import os

# Remonte d'un dossier vers la racine du projet (le parent)
chemin_racine = os.path.abspath(os.path.join(os.getcwd(), '..'))

# Ajoute ce chemin s'il n'y est pas déjà
if chemin_racine not in sys.path:
    sys.path.append(chemin_racine)

import gf_model.GFmodel as gf

fold_num = 1
file_num = 28
file_num_str = str(file_num).zfill(2)

# path to LES files
LES_path = '/LES/'

In [11]:
import numpy as np
import torch
import gf_model.GFmodel as gf
import PPmodel as pp

def generate_ktf17_dataset_via_logpost(num_samples=10000, save_path="ktf17_dataset.pt"):
    thetas = []
    xs = []
    
    # A. Observation (y_obs)
    noisy_lc = np.loadtxt((chemin_racine + LES_path + 'mean/%d_' + file_num_str + '_infall.txt') %fold_num , delimiter = ',')
    noisy_extrem = (np.argmax(noisy_lc)) * 20
    
    # B. Matrice de sous-échantillonnage H
    ny = len(noisy_lc)
    nx = (len(noisy_lc) - 1) * 20 + 1
    spacing = 20
    H = np.zeros((ny, nx))
    for i in range(0, ny): 
        H[i, i*spacing] = 1
        
    # C. Matrices SVD pour le calcul de la log-probabilité
    R = np.loadtxt((chemin_racine + LES_path + 'cov/%d_' + file_num_str + '_infall.txt') %fold_num, delimiter = ',')
    u, l, q = np.linalg.svd(R)
    
    print(f"Préparation terminée. Génération de {num_samples} échantillons en cours...")
    
    rejets = 0 # On ajoute un compteur pour surveiller
    
    # --- 2. Boucle de génération ---
    while len(thetas) < num_samples:
        
        H0 = np.random.uniform(0, 4000)
        tau = np.random.uniform(0, 720)
        T = np.random.uniform(0, tau)
        N = np.random.uniform(0, 1e8)
        
        samp = [H0, tau, T, N]

        # On récupère le résultat
        log_prob, M_retenu = pp.log_post(samp, noisy_lc, noisy_extrem, nx, u, l, H, with_M=True)

        # Si le set de paramètres est valide (le prior a accepté et un cycle a été trouvé)
        if log_prob > -np.inf and M_retenu is not None:
            thetas.append(samp)
            xs.append(M_retenu)
            
            if len(thetas) % 100 == 0: # Affichage tous les 10 pour voir que ça avance
                print(f"{len(thetas)} / {num_samples} générés... (Rejets cumulés : {rejets})")
        else:
            rejets += 1

    # --- 3. Sauvegarde au format PyTorch pour Effortless SBI ---
    dataset = {
        'theta': torch.tensor(thetas, dtype=torch.float32),
        'x': torch.tensor(xs, dtype=torch.float32),
        'y_obs': torch.tensor(noisy_lc, dtype=torch.float32)
    }
    torch.save(dataset, save_path)
    print(f"Terminé ! Dataset sauvegardé sous : {save_path}")

In [ ]:
import numpy as np
import torch
import gf_model.GFmodel as gf
import PPmodel as pp

def generate_ktf17_dataset_via_prior(num_samples=10000, save_path="ktf17_dataset.pt"):
    thetas = []
    xs = []
    noisy_lc = []
    count = 0

    # --- 2. Boucle de génération ---
    while len(thetas) < num_samples:
        
        uniform = np.zeros(4)
        uniform[0] = np.random.uniform(0, 4000) #H_0
        uniform[1] = np.random.uniform(0, 720) #tau in minutes
        uniform[2] = np.random.uniform(0, 720) #T 
        uniform[3] = np.random.uniform(0, 1e8) #N
        if pp.prior(uniform) == 1: 
            thetas.append(uniform)
            count+=1
            if count%100 == 0 : 
                print(count)

    # --- 3. Sauvegarde au format PyTorch pour Effortless SBI ---
    dataset = {
        'theta': torch.tensor(thetas, dtype=torch.float32),
        'x': torch.tensor(xs, dtype=torch.float32),
        'y_obs': torch.tensor(noisy_lc, dtype=torch.float32)
    }
    torch.save(dataset, save_path)
    print(f"Terminé ! Dataset sauvegardé sous : {save_path}")

In [31]:
import numpy as np
import torch
import gf_model.GFmodel as gf
import PPmodel as pp
from scipy.special import lambertw
import scipy.signal

def generate_ktf17_dataset_optimal(num_samples=10000, save_path="ktf17_dataset.pt"):
    thetas = []
    xs = []
    
    # A. Observation (y_obs) et Matrices
    noisy_lc = np.loadtxt((chemin_racine + LES_path + 'mean/%d_' + file_num_str + '_infall.txt') %fold_num , delimiter = ',')
    noisy_extrem = (np.argmax(noisy_lc)) * 20
    
    ny = len(noisy_lc)
    nx = (ny - 1) * 20 + 1
    spacing = 20
    H_in = np.zeros((ny, nx))
    for i in range(0, ny): 
        H_in[i, i*spacing] = 1
        
    print(f"Génération stricte (Indépendante) de {num_samples} échantillons...")
    
    count_tries = 0
    
    while len(thetas) < num_samples:
        count_tries += 1
        
        # 1. Tirage STRICTEMENT INDÉPENDANT (Comme dans via_prior)
        H0 = np.random.uniform(0, 4000)
        tau = np.random.uniform(0, 720)
        T = np.random.uniform(0, 720)
        N = np.random.uniform(0, 1e8)
        samp = [H0, tau, T, N]
        
        # 2. Filtres mathématiques rapides (Avant de lancer RK4)
        if T >= tau:
            continue
            
        mu = N / ((1e6/1440) * tau * H0)
        in_w = (-2) * (np.sqrt((1/mu) + 0.25) - 0.5) * (T/tau) * np.exp(T/tau)
        beta = (tau/T) * lambertw(in_w) - 1
        
        if beta.real <= 0:
            continue
            
        # 3. Exécution de RK4 UNE SEULE FOIS
        dt = 0.1
        solve_time = 1440
        Hint = 0.1 * np.ones(len(np.arange(0, T + dt, dt)))
        
        model_run = pp.dim_RK4(H0, tau, T, N, Hint, solve_time, dt) 
        myLimitCycle = pp.FindLimitCycle(model_run, samp)
        
        # Vérification du cycle limite
        if (any(m < 0 for m in myLimitCycle) or not np.any(scipy.signal.argrelextrema(myLimitCycle, np.greater)[0])):
            continue
            
        # 4. Alignement direct (Sans repasser par log_post)
        M = myLimitCycle
        lHd = nx
        Hd_max = noisy_extrem
        lM = len(M)
        M_max = np.argmax(M)
        
        if (lM - M_max > lHd - Hd_max): 
            M = M[:M_max + lHd - Hd_max]
        else: 
            M = np.concatenate((M, np.zeros((lHd - Hd_max) - (lM - M_max))))
        
        if (M_max > Hd_max): 
            M = M[M_max - Hd_max:]
        else: 
            M = np.concatenate((np.zeros(Hd_max - M_max), M))
            
        M_aligne = H_in @ M
        
        # Ajout au Dataset
        thetas.append(samp)
        xs.append(M_aligne)
        
        if len(thetas) % 100 == 0:
            print(f"{len(thetas)} / {num_samples} générés... (Tirages totaux : {count_tries})")

    # --- Sauvegarde ---
    dataset = {
        'theta': torch.tensor(thetas, dtype=torch.float32),
        'x': torch.tensor(xs, dtype=torch.float32),
        'y_obs': torch.tensor(noisy_lc, dtype=torch.float32)
    }
    torch.save(dataset, save_path)
    print(f"Dataset non-biaisé sauvegardé sous : {save_path}")

In [32]:
generate_ktf17_dataset_optimal(num_samples=10000, save_path="ktf17_dataset_10000_opti.pt")

Génération stricte (Indépendante) de 10000 échantillons...
100 / 10000 générés... (Tirages totaux : 3906)
200 / 10000 générés... (Tirages totaux : 7113)
300 / 10000 générés... (Tirages totaux : 10463)
400 / 10000 générés... (Tirages totaux : 14064)
500 / 10000 générés... (Tirages totaux : 18160)
600 / 10000 générés... (Tirages totaux : 21337)
700 / 10000 générés... (Tirages totaux : 24822)
800 / 10000 générés... (Tirages totaux : 28965)
900 / 10000 générés... (Tirages totaux : 33331)
1000 / 10000 générés... (Tirages totaux : 37410)
1100 / 10000 générés... (Tirages totaux : 40975)
1200 / 10000 générés... (Tirages totaux : 44441)
1300 / 10000 générés... (Tirages totaux : 48475)
1400 / 10000 générés... (Tirages totaux : 52554)
1500 / 10000 générés... (Tirages totaux : 56534)
1600 / 10000 générés... (Tirages totaux : 60490)
1700 / 10000 générés... (Tirages totaux : 64064)
1800 / 10000 générés... (Tirages totaux : 67959)
1900 / 10000 générés... (Tirages totaux : 71830)
2000 / 10000 générés.

In [33]:
import torch
import numpy as np

def add_noise_to_dataset(dataset_path="ktf17_dataset.pt", save_path="ktf17_dataset_noisy.pt"):
    """
    Charge le dataset généré, ajoute un bruit multivarié basé sur la matrice de covariance R,
    et enregistre le nouveau dataset avec la clé 'x_noisy'.
    """
    # 1. Charger le dataset existant
    dataset = torch.load(dataset_path)
    xs = dataset['x'] # Shape: (N_samples, 121)
    
    R_path = (chemin_racine + LES_path + 'cov/%d_' + file_num_str + '_infall.txt') % fold_num
    R = np.loadtxt(R_path, delimiter=',')
    
    # 3. Correction du papier : "Inflating the covariance diagonal"
    # Les auteurs ajoutent 10% de la variance maximale à toute la diagonale
    max_variance = np.max(np.diag(R))
    R_inflated = R + 0.1 * max_variance * np.eye(len(R))
    
    # Sécurité numérique : s'assurer que la matrice est parfaitement symétrique
    R_inflated = (R_inflated + R_inflated.T) / 2
    
    # 4. Génération du bruit Gaussien pour chaque simulation
    N_samples = xs.shape[0]
    dim_x = xs.shape[1]
    
    print(f"Génération du bruit pour {N_samples} échantillons...")
    
    # np.random.multivariate_normal tire des échantillons de N(0, R_inflated)
    noise = np.random.multivariate_normal(mean=np.zeros(dim_x), cov=R_inflated, size=N_samples)
    
    # 5. Création de x_noisy
    noise_tensor = torch.tensor(noise, dtype=torch.float32)
    x_noisy = xs + noise_tensor
    
    # 6. Mise à jour du dictionnaire et sauvegarde
    dataset['x_noisy'] = x_noisy
    torch.save(dataset, save_path)
    
    print(f"Terminé ! Le dataset bruité a été sauvegardé sous : {save_path}")

# Utilisation :
add_noise_to_dataset("ktf17_dataset_10000_opti.pt", "ktf17_dataset_10000_opti_noisy.pt")

Génération du bruit pour 10000 échantillons...
Terminé ! Le dataset bruité a été sauvegardé sous : ktf17_dataset_10000_opti_noisy.pt


# MCMC

In [5]:
import numpy as np
import torch
import emcee
import PPmodel as pp

# --- 1. Chargement de l'observation cible depuis le dataset ---
dataset = torch.load("ktf17_dataset_10000.pt")

# On extrait le 1er échantillon (index 0)
target_idx = 0
noisy_lc = dataset['x'][target_idx].numpy()        # La simulation devient notre cible
theta_true = dataset['theta'][target_idx].numpy()  # La vérité terrain à retrouver

# Variables nécessaires pour log_post
noisy_extrem = (np.argmax(noisy_lc)) * 20
ny = len(noisy_lc)
nx = (ny - 1) * 20 + 1
spacing = 20
H = np.zeros((ny, nx))
for i in range(0, ny): 
    H[i, i*spacing] = 1

R = np.loadtxt((chemin_racine + LES_path + 'cov/%d_' + file_num_str + '_infall.txt') %fold_num, delimiter = ',')
u, l, q = np.linalg.svd(R)

# --- 2. Initialisation du MCMC ---
nwalkers = 20
ndim = 4
exp_num = '_exp01'

print("Initialisation des walkers...")
my_samples = np.loadtxt((chemin_racine + '/pp_model/files/prior/prior_samples' + exp_num + '.txt'), delimiter = ',')

my_random_samples = np.random.randint(0, len(my_samples), 5000)
my_log_probs = np.zeros(len(my_random_samples))

for tt in range(0, len(my_log_probs)): 
    testing = my_samples[my_random_samples[tt]]
    # Appel de log_post classique (sans with_M pour avoir juste le score)
    my_log_probs[tt] = pp.log_post(testing, noisy_lc, noisy_extrem, nx, u, l, H)

# Sélection des 20 meilleurs points pour démarrer
x_samples = my_samples[my_random_samples[np.argsort(my_log_probs)[-nwalkers:]]]

# --- 3. Lancement de emcee ---
# Optionnel: on garde le backend HDF5 des auteurs pour la sauvegarde temps réel
filename = chemin_racine + '/pp_model/files/mcmc_runs/test.h5'
backend = emcee.backends.HDFBackend(filename)
backend.reset(nwalkers, ndim)

sampler = emcee.EnsembleSampler(
    nwalkers, 
    ndim, 
    pp.log_post, 
    args=[noisy_lc, noisy_extrem, nx, u, l, H],  
    backend=backend
)

# On lance le MCMC (500 étapes pour le test, mettez 5000+ pour des vrais résultats)
n_steps = 500
print(f"Lancement de {n_steps} étapes de MCMC...")
sampler.run_mcmc(x_samples, n_steps, progress=True)

# --- 4. Extraction et Sauvegarde ---
# On ignore les 100 premières étapes (burn-in) et on aplatit les walkers
flat_samples = sampler.get_chain(discard=100, flat=True)
np.save("mcmc_samples_target0.npy", flat_samples)
print("Données MCMC sauvegardées dans 'mcmc_samples_target0.npy'")

Initialisation des walkers...


You must install the tqdm library to use progress indicators with emcee


Lancement de 500 étapes de MCMC...
Données MCMC sauvegardées dans 'mcmc_samples_target0.npy'


In [6]:
import numpy as np
import emcee
import PPmodel as pp

# Chargement direct du fichier texte
noisy_lc = np.loadtxt((chemin_racine + LES_path + 'mean/%d_' + file_num_str + '_infall.txt') %fold_num , delimiter = ',')

# Variables nécessaires pour log_post
noisy_extrem = (np.argmax(noisy_lc)) * 20
ny = len(noisy_lc)
nx = (ny - 1) * 20 + 1
spacing = 20
H = np.zeros((ny, nx))
for i in range(0, ny): 
    H[i, i*spacing] = 1

# Rechargement de la matrice de covariance
R = np.loadtxt((chemin_racine + LES_path + 'cov/%d_' + file_num_str + '_infall.txt') %fold_num, delimiter = ',')
u, l, q = np.linalg.svd(R)

# --- 2. Initialisation du MCMC ---
nwalkers = 20
ndim = 4
exp_num = '_exp01'

print("Initialisation des walkers...")
my_samples = np.loadtxt((chemin_racine + '/pp_model/files/prior/prior_samples' + exp_num + '.txt'), delimiter = ',')

my_random_samples = np.random.randint(0, len(my_samples), 5000)
my_log_probs = np.zeros(len(my_random_samples))

for tt in range(0, len(my_log_probs)): 
    testing = my_samples[my_random_samples[tt]]
    # Appel de log_post classique (sans with_M pour avoir juste le score)
    my_log_probs[tt] = pp.log_post(testing, noisy_lc, noisy_extrem, nx, u, l, H)

# Sélection des 20 meilleurs points pour démarrer
x_samples = my_samples[my_random_samples[np.argsort(my_log_probs)[-nwalkers:]]]

# --- 3. Lancement de emcee ---
# Optionnel: on garde le backend HDF5 des auteurs pour la sauvegarde temps réel
filename = chemin_racine + '/pp_model/files/mcmc_runs/test.h5'
backend = emcee.backends.HDFBackend(filename)
backend.reset(nwalkers, ndim)

sampler = emcee.EnsembleSampler(
    nwalkers, 
    ndim, 
    pp.log_post, 
    args=[noisy_lc, noisy_extrem, nx, u, l, H],  
    backend=backend
)

# On lance le MCMC (500 étapes pour le test, mettez 5000+ pour des vrais résultats)
n_steps = 500
print(f"Lancement de {n_steps} étapes de MCMC...")
sampler.run_mcmc(x_samples, n_steps, progress=True)

# --- 4. Extraction et Sauvegarde ---
# On ignore les 100 premières étapes (burn-in) et on aplatit les walkers
flat_samples = sampler.get_chain(discard=100, flat=True)
np.save("mcmc_samples_targety.npy", flat_samples)
print("Données MCMC sauvegardées dans 'mcmc_samples_targety.npy'")

Initialisation des walkers...


You must install the tqdm library to use progress indicators with emcee


Lancement de 500 étapes de MCMC...
Données MCMC sauvegardées dans 'mcmc_samples_targety.npy'
